In [1]:
OPENAI_API_KEY = ""

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
from openai import OpenAI
import json
from typing import Dict

client = OpenAI(api_key = OPENAI_API_KEY)

# Load the spreadsheet
file_path = './school_and_program.xlsx'
spreadsheet = pd.ExcelFile(file_path)

# Load Sheet4
sheet4 = pd.read_excel(spreadsheet, 'links')


In [4]:
ap_courses = None
file_path = "./ap.json"
with open(file_path, "r", encoding="utf-8") as file:
    ap_courses = json.load(file)
# print(ap_courses)

course_dict = {}

if ap_courses and "data" in ap_courses:
    for course in ap_courses["data"]:
        course_dict[course["en_name"]] = {
            "en_name": course["en_name"],
            "course_code": course["course_code"],
            "course_system_subject": course["course_system_subject"],
            "uforse_subject": course["uforse_subject"],
        }

In [5]:
class GPT_Wrapper:
    fetch_sys_prompt = """
    # instruction
    you need to fetch the data about a university program from the URL that I give you
    # Format:
        If you can't find anything, just return an empty string

        For TOEFL, return only marks of iBT, like "TOEFL: 80"
        
        for language requirement (TOEFL/IELTS/Duolingo), format:
            name: score
            Extra information on the next line if there is any

    # example:
    ## target
        Ontario Admission Requirement
    ## website
        https://algomau.ca/admissions/admissions-requirements/
    ## program
        Biology
    ## expect output: string separated by enter
        ENG4U\nMHF4U\nTwo U/M level sciences (Biology and Chemistry recommended)\nMinimum average: 70%

    return only a string, without further explanation
    
    From now on, I will only give you target, website and program for you to do the work
    """
    
    ap_sys_prompt = prompt = f"""
    # instruction
    you are a high school admin that helps student to select the AP courses while applying for the universities.
    Based on other application requirement from the university, you need to select the most relevant AP course for them.
    The content presented should be as comprehensive as possible.
    Currently, you can choose among any of the ap courses:

    {json.dumps(course_dict)}

    # Format
    Input: Enter separated description of requirements
        Program
        ON Admission Requirement
        IB Admission Requirement
        BC Admission Requirement
        AP Admission Requirement
    Output: Enter separated strings of AP course recommended. 
        No need to format or number the string at all, just return the result.
        `course_name` (course_code): `expected grade if there exists`.
        Any other general requirement about AP may follow.
        
    # Example
    ## Input
    Program: Economics
    ON Admission Requirement:
        ENG4U
        Calculus & Vectors (MCV4U)
        Advanced Functions (MHF4U)
        Minimum average: mid-80s
    IB Admission Requirement:
        International Baccalaureate Diploma, with a total score of at least 30 including bonus points. 
        Mathematics: Analysis and Approaches HL or SL, or Mathematics: Applications and Interpretation HL. 
        Minimum scores of 4 in each subject.
    BC Admission Requirement:
        English (ENG4U)
        Calculus and Vectors (MCV4U)
        Advanced Functions (MHF4U)
        Two additional 4U/M courses
        Minimum average: mid-80s
    AP Admission Requirement:
        Advanced Level passes in the following subjects:
            - Economics
            - Mathematics
    
    ## Output
        AP English Language and Composition (AP12ELC): 4+ 
        AP Calculus AB (AP12CA): 4+
        AP Calculus BC (AP12CB): 4+
        AP Statistics (AP12SLC): 4+
        AP Macroeconomics (AP12MAE): 4+
        AP Microeconomics (AP12MIE): 4+
        
    # Notice
    You only need to generate based on the content that I find for you, and don't find anything else from the internet
    If you have trouble generate, return empty string instead.
    
    From now on, I will only give you the json of content for you to do the work
    """
    
    def __init__(self) -> None:
        self.fetch_client = OpenAI(api_key = OPENAI_API_KEY)
        self.ap_client = OpenAI(api_key = OPENAI_API_KEY)
        # _ = self.fetch_client.chat.completions.create(
        #     model="gpt-4o-2024-05-13",
        #     messages=[{"role":"system", "content": self.fetch_sys_prompt}],
        #     temperature=0
        # )
        # _ = self.ap_client.chat.completions.create(
        #     model="gpt-4o-2024-05-13",
        #     messages=[{"role":"system", "content": self.ap_sys_prompt}],
        #     temperature=0
        # )

    def fetch_admission_requirement(self, target, program, url):
        response = self.fetch_client.chat.completions.create(
            model="gpt-4o-2024-05-13",
            messages=[{"role":"system", "content": self.fetch_sys_prompt}, {"role": "user", "content": json.dumps({"target": target, "website": url, "program": program})}],
            temperature=0
        )
        return response.choices[0].message.content

    def suggest_ap_courses(self, admin_dict: Dict[str, str]) -> str:
        response = self.ap_client.chat.completions.create(
            model="gpt-4o-2024-05-13",
            messages=[{"role":"system", "content": self.ap_sys_prompt}, {"role": "user", "content": json.dumps(admin_dict)}],
            temperature=0
        )
        return response.choices[0].message.content

In [6]:

# Define the function to fetch the data from the URL
def fetch_admission_requirement(target, program, url):
    prompt = """
    # instruction
    you need to fetch the data about a university program from the URL that I give you
    # Format:
        If you can't find anything, just return an empty string

        For TOEFL, return only marks of iBT, like "TOEFL: 80"
        
        for language requirement (TOEFL/IELTS/Duolingo), format:
            name: score
            Extra information on the next line if there is any

    # example:
    ## target
        Ontario Admission Requirement
    ## website
        https://algomau.ca/admissions/admissions-requirements/
    ## program
        Biology
    ## expect output: string separated by enter
        ENG4U\nMHF4U\nTwo U/M level sciences (Biology and Chemistry recommended)\nMinimum average: 70%

    return only a string, without further explanation
    
    From now on, I will only give you target, website and program for you to do the work
    """
    response = client.chat.completions.create(
        model="gpt-4o-2024-05-13",
        messages=[{"role":"system", "content": prompt}, {"role": "user", "content": json.dumps({"target": target, "website": url, "program": program})}],
        temperature=0
    )
    return response.choices[0].message.content


In [7]:


def suggest_ap_gpt(admin_dict: Dict[str, str]) -> str:
    prompt = f"""
    # instruction
    you are a high school admin that helps student to select the AP courses while applying for the universities.
    Based on other application requirement from the university, you need to select the most relevant AP course for them.
    The content presented should be as comprehensive as possible.
    Currently, you can choose among any of the ap courses:

    {json.dumps(course_dict)}

    # Format
    Input: Enter separated description of requirements
        Program
        ON Admission Requirement
        IB Admission Requirement
        BC Admission Requirement
        AP Admission Requirement
    Output: Enter separated strings of AP course recommended. 
        No need to format or number the string at all, just return the result.
        `course_name` (course_code): `expected grade if there exists`
        Any other general requirement about AP may follow.
        
    # Example
    ## Input
    Program: Economics
    ON Admission Requirement:
        ENG4U
        Calculus & Vectors (MCV4U)
        Advanced Functions (MHF4U)
        Minimum average: mid-80s
    IB Admission Requirement:
        International Baccalaureate Diploma, with a total score of at least 30 including bonus points. 
        Mathematics: Analysis and Approaches HL or SL, or Mathematics: Applications and Interpretation HL. 
        Minimum scores of 4 in each subject.
    BC Admission Requirement:
        English (ENG4U)
        Calculus and Vectors (MCV4U)
        Advanced Functions (MHF4U)
        Two additional 4U/M courses
        Minimum average: mid-80s
    AP Admission Requirement:
        Advanced Level passes in the following subjects:
            - Economics
            - Mathematics
    
    ## Output
        AP English Language and Composition (AP12ELC): 4+ 
        AP Calculus AB (AP12CA): 4+
        AP Calculus BC (AP12CB): 4+
        AP Statistics (AP12SLC): 4+
        AP Macroeconomics (AP12MAE): 4+
        AP Microeconomics (AP12MIE): 4+
    """
    response = client.chat.completions.create(
        model="gpt-4o-2024-05-13",
        messages=[{"role":"system", "content": prompt}, {"role": "user", "content": json.dumps(admin_dict)}],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
# suggest_ap_gpt()

In [8]:
res = fetch_admission_requirement("IELTS", "Sociology", "https://calendar.carleton.ca/undergrad/regulations/admissions/general/")
print(res)

IELTS: 6.5
Minimum score of 6.0 in each band


In [ ]:
len(sheet4)

In [ ]:
# Create a list to store the result data
result_data = []

# Loop through the dataframe and fetch the requirements
target_attributes = [
    "ON Admission Requirement",
    "IB Admission Requirement",
    "BC Admission Requirement",
    "AP Admission Requirement",
    "Language Requirement",
]
lang_attributes = ["IELTS", "TOEFL", "Duolingo"]

my_gpt = GPT_Wrapper()

count = 0
# [17:25]
for index, row in sheet4.iterrows():
    university = row["University"]
    program = row["Program and link"]
    # on_admission_url = row['ON Admission Requirement']
    # ib_admission_url = row['IB Admission Requirement']
    # bc_admission_url = row['BC Admission Requirement']
    # ap_admission_url = row['AP Admission Requirement']
    # language_requirement_url = row['Language Requirement']
    # print(row)
    result_row = {"University": university, "Program": program}
    for attr in target_attributes:
        print(attr, program, row[attr])
        if pd.notna(row[attr]):
            if attr == "Language Requirement":
                for lang_attr in lang_attributes:
                    requirements = my_gpt.fetch_admission_requirement(lang_attr, program, row[attr])
                    result_row[lang_attr] = requirements
                continue
            requirements = my_gpt.fetch_admission_requirement(attr, program, row[attr])
            # print(requirements)
            # result_data.append({'University': university, 'Program': program, 'Target': attr, 'Requirements': requirements})
            result_row[attr] = requirements
        if attr == "AP Admission Requirement":
            input_dict = {}
            input_dict["Program"] = program
            input_dict["ON Admission Requirement"] = (
                result_row["ON Admission Requirement"] if "ON Admission Requirement" in result_row else ""
            )
            input_dict["IB Admission Requirement"] = (
                result_row["IB Admission Requirement"] if "IB Admission Requirement" in result_row else ""
            )
            input_dict["BC Admission Requirement"] = (
                result_row["BC Admission Requirement"] if "BC Admission Requirement" in result_row else ""
            )
            input_dict["AP Admission Requirement"] = (
                result_row["AP Admission Requirement"] if "AP Admission Requirement" in result_row else ""
            )
            ap_admin = my_gpt.suggest_ap_courses(input_dict)
            # print(ap_admin)
            result_row[attr] = ap_admin
        if attr in result_row:
            print(result_row[attr])
    if len(result_row) >= 3:
        result_data.append(result_row)

    # Repeat for other targets if URLs are provided

In [ ]:
# Convert the result data to a dataframe
result_df = pd.DataFrame(result_data)

# Show the result dataframe
result_df.head()


In [ ]:
result_df.to_csv("./admin_requirements.csv", index=False)

In [10]:
on_course = {
    "data": [
        {
            "id": 212,
            "ch_name": "12年级加拿大历史",
            "en_name": "G12 Canadian History",
            "course_system_id": 2,
            "course_code": "ON12CHI",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.553Z",
            "updated_at": "2024-03-27T15:56:49.541Z",
            "course_system_subject": "Other",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 211,
            "ch_name": "12年级世界问题:地理分析",
            "en_name": "G12 World Issues: Geographic  Analysis ",
            "course_system_id": 2,
            "course_code": "ON12WIGA",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.545Z",
            "updated_at": "2024-03-27T15:39:21.828Z",
            "course_system_subject": "Other",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 213,
            "ch_name": "12年级世界历史",
            "en_name": "G12 World History",
            "course_system_id": 2,
            "course_code": "ON12WH",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.558Z",
            "updated_at": "2024-03-27T15:38:58.271Z",
            "course_system_subject": "Other",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 215,
            "ch_name": "12年级英语",
            "en_name": "G12 English",
            "course_system_id": 2,
            "course_code": "ON12EN",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.566Z",
            "updated_at": "2024-03-27T15:38:50.836Z",
            "course_system_subject": "Other",
            "uforse_subject": "English",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 217,
            "ch_name": "12年级作家的技巧",
            "en_name": "G12 The Writer's Craft",
            "course_system_id": 2,
            "course_code": "ON12WC",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.576Z",
            "updated_at": "2024-03-27T15:38:43.494Z",
            "course_system_subject": "Other",
            "uforse_subject": "English",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 220,
            "ch_name": "12年级数据管理",
            "en_name": "G12 Data Management",
            "course_system_id": 2,
            "course_code": "ON12DM",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.589Z",
            "updated_at": "2024-03-27T15:38:33.102Z",
            "course_system_subject": "Mathematics",
            "uforse_subject": "Mathematics",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 224,
            "ch_name": "12年级运动学概论",
            "en_name": "G12 Introduction to Kinesiology",
            "course_system_id": 2,
            "course_code": "ON12IK",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.607Z",
            "updated_at": "2024-03-27T15:38:15.065Z",
            "course_system_subject": "Sciences",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 225,
            "ch_name": "12年级生物",
            "en_name": "G12 Biology",
            "course_system_id": 2,
            "course_code": "ON12BI",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.611Z",
            "updated_at": "2024-03-27T15:36:16.740Z",
            "course_system_subject": "Sciences",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 226,
            "ch_name": "12年级化学",
            "en_name": "G12 Chemistry",
            "course_system_id": 2,
            "course_code": "ON12CH",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.615Z",
            "updated_at": "2024-03-27T15:36:05.198Z",
            "course_system_subject": "Sciences",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        },
        {
            "id": 227,
            "ch_name": "12年级物理",
            "en_name": "G12 Physics",
            "course_system_id": 2,
            "course_code": "ON12PH",
            "grade_level": "12",
            "prerequisite": None,
            "extra_academic_test": None,
            "full_mark": "100",
            "course_description": None,
            "created_at": "2023-08-18T14:20:56.620Z",
            "updated_at": "2024-03-27T15:35:53.477Z",
            "course_system_subject": "Sciences",
            "uforse_subject": "Other",
            "course_system": {
                "id": 2,
                "en_name": "Ontario Curriculum"
            }
        }
    ],
    "total": 68
}


In [12]:
cleaned_on_course = []
for course in on_course["data"]:
    cleaned_on_course.append({"en_name": course["en_name"], "course_code": course["course_code"]})

In [13]:
print(cleaned_on_course)

[{'en_name': 'G12 Canadian History', 'course_code': 'ON12CHI'}, {'en_name': 'G12 World Issues: Geographic  Analysis ', 'course_code': 'ON12WIGA'}, {'en_name': 'G12 World History', 'course_code': 'ON12WH'}, {'en_name': 'G12 English', 'course_code': 'ON12EN'}, {'en_name': "G12 The Writer's Craft", 'course_code': 'ON12WC'}, {'en_name': 'G12 Data Management', 'course_code': 'ON12DM'}, {'en_name': 'G12 Introduction to Kinesiology', 'course_code': 'ON12IK'}, {'en_name': 'G12 Biology', 'course_code': 'ON12BI'}, {'en_name': 'G12 Chemistry', 'course_code': 'ON12CH'}, {'en_name': 'G12 Physics', 'course_code': 'ON12PH'}]


In [34]:

def suggest_on_course(profile_dict: Dict[str, str], target_program: Dict[str, str]) -> str:
    client = OpenAI(api_key = OPENAI_API_KEY)
    
    prompt = f"""
    # System prompt
    You are given admin requirement to apply for a program, and a profile of a student.
    You should return empty string if the student already fulfill the requirement, or suggest the courses to take in Ontario curriculum.
    Please present all possible choices
    Note: here is a reference of all courses that you can choose from, do not include anything outside here:
    {cleaned_on_course}
    You do not need to number the course in any way
    # Output format
    just return: `number of courses to take`
    `en_name`, `course_code`
    separated by new lines
    """
    
    user_content = json.dumps({"profile": profile_dict, "target_program": target_program})
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-05-13",
        messages=[{"role":"system", "content": prompt}, {"role": "user", "content": user_content }],
        temperature=0
    )
    return response.choices[0].message.content

In [17]:
profile1 = {
    "id": 163,
    "student_id": 134,
    "country_id": 1,
    "program_category_id": 4,
    "university_id": 12,
    "program_id": 10,
    "aspiration": None,
    "interests": None,
    "high_school": "abcc",
    "course_system_id": 2,
    "grade": "Grade 12",
    "vol_hours": 0,
    "created_at": "2024-03-02T13:36:03.826Z",
    "updated_at": "2024-03-10T05:06:44.634Z",
    "career_status_at_30": None,
    "if_need_language_test": True,
    "master_student_profile_modification_log_id": 234,
    "student": {"id": 134, "name": "william"},
    "country": {"id": 1, "ch_name": "加拿大", "en_name": "Canada"},
    "program_category": {"id": 4, "ch_name": "计算机\t", "en_name": "Computer Sci"},
    "university": {
        "id": 12,
        "ch_name": "多伦多大学\t\t",
        "en_name": "University of Toronto\t",
        "offer_logo": {
            "id": 111,
            "src": "https://www.applyintelligence.tech/rails/active_storage/blobs/redirect/eyJfcmFpbHMiOnsibWVzc2FnZSI6IkJBaHBkQT09IiwiZXhwIjpudWxsLCJwdXIiOiJibG9iX2lkIn19--c37bdd27b31af7ae11837d74bb812ee2e41efbe2/logo-uot.jpg",
        },
    },
    "program": {
        "id": 10,
        "ch_name": "多伦多大学计算机学院\t\t",
        "en_name": "U of T Computer Science undergraduate",
        "course_weight": 0.6,
        "extra_curriculum_weight": 0.1,
        "leadership_weight": 0.3,
        "course_threshold": 98,
        "extra_curriculum_threshold": 100,
        "leadership_threshold": 80,
    },
    "course_system": {"id": 2, "ch_name": "安大略省的课程(ON)\t", "en_name": "Ontario Curriculum"},
    "student_profile_courses": [
        {
            "id": 17,
            "course_id": 308,
            "score": 88,
            "course": {"id": 308, "ch_name": "综合艺术", "en_name": "Integrated Arts", "full_mark": "100"},
        },
        {
            "id": 18,
            "course_id": 294,
            "score": 99,
            "course": {"id": 294, "ch_name": "音乐", "en_name": "Music", "full_mark": "100"},
        },
        {
            "id": 27,
            "course_id": 313,
            "score": 55,
            "course": {"id": 313, "ch_name": "视觉艺术", "en_name": "Visual Arts", "full_mark": "100"},
        },
        {
            "id": 26,
            "course_id": 313,
            "score": 58,
            "course": {"id": 313, "ch_name": "视觉艺术", "en_name": "Visual Arts", "full_mark": "100"},
        },
        {
            "id": 32,
            "course_id": 296,
            "score": 55,
            "course": {"id": 296, "ch_name": "商科入门", "en_name": "Introduction to Business", "full_mark": "100"},
        },
    ],
    "student_profile_language_tests": [
        {
            "id": 28,
            "language_test_id": 2,
            "language_test": {"id": 2, "ch_name": "托福\t", "en_name": "TOEFL\t"},
            "student_profile_language_test_language_test_parts": [
                {
                    "id": 169,
                    "language_test_part_id": 6,
                    "score": None,
                    "language_test_part": {"id": 6, "ch_name": "托福写作", "en_name": "TOEFL Writing"},
                },
                {
                    "id": 170,
                    "language_test_part_id": 5,
                    "score": None,
                    "language_test_part": {"id": 5, "ch_name": "托福阅读 ", "en_name": "TOEFL Reading "},
                },
                {
                    "id": 171,
                    "language_test_part_id": 4,
                    "score": None,
                    "language_test_part": {"id": 4, "ch_name": "托福口语\t", "en_name": "TOEFL Speaking\t"},
                },
                {
                    "id": 172,
                    "language_test_part_id": 1,
                    "score": None,
                    "language_test_part": {"id": 1, "ch_name": "托福听力", "en_name": "TOEFL Listening"},
                },
            ],
        },
        {
            "id": 29,
            "language_test_id": 2,
            "language_test": {"id": 2, "ch_name": "托福\t", "en_name": "TOEFL\t"},
            "student_profile_language_test_language_test_parts": [
                {
                    "id": 173,
                    "language_test_part_id": 6,
                    "score": None,
                    "language_test_part": {"id": 6, "ch_name": "托福写作", "en_name": "TOEFL Writing"},
                },
                {
                    "id": 174,
                    "language_test_part_id": 5,
                    "score": None,
                    "language_test_part": {"id": 5, "ch_name": "托福阅读 ", "en_name": "TOEFL Reading "},
                },
                {
                    "id": 175,
                    "language_test_part_id": 4,
                    "score": None,
                    "language_test_part": {"id": 4, "ch_name": "托福口语\t", "en_name": "TOEFL Speaking\t"},
                },
                {
                    "id": 176,
                    "language_test_part_id": 1,
                    "score": None,
                    "language_test_part": {"id": 1, "ch_name": "托福听力", "en_name": "TOEFL Listening"},
                },
            ],
        },
        {"id": 30, "language_test_id": None, "student_profile_language_test_language_test_parts": []},
    ],
    "student_profile_competitions": [
        {
            "id": 14,
            "competition_subject": "Chemistry",
            "competition_name": "abbb",
            "award_level": None,
            "award_name_or_ranking": "",
            "start_time": None,
            "end_time": None,
        }
    ],
    "student_profile_standardized_tests": [],
    "student_profile_certificates": [],
    "student_profile_summer_schools": [
        {"id": 10, "name": "xab", "summer_school_name": "", "start_time": None, "end_time": None},
        {"id": 11, "name": "abb", "summer_school_name": "xxhh", "start_time": None, "end_time": None},
        {"id": 9, "name": "xabbb", "summer_school_name": "", "start_time": None, "end_time": None},
    ],
    "student_profile_research_experiments": [],
    "student_profile_volunteers": [],
    "student_profile_extracurriculars": [
        {
            "id": 10,
            "extracurricular_type": "Case Competition",
            "club_type": None,
            "club_members": None,
            "if_founder": False,
            "if_leader": False,
            "if_than_year": False,
            "start_time": None,
            "end_time": None,
        }
    ],
    "student_profile_leaderships": [],
    "student_profile_team_competitions": [],
    "student_profile_current_courses": [],
}

In [18]:
df = pd.read_csv("./admin_requirements.csv")

In [19]:
df.head()

,University,Program,ON Admission Requirement,IB Admission Requirement,BC Admission Requirement,AP Admission Requirement,IELTS,TOEFL,Duolingo
0,Algoma University,Biology,ENG4U\nMHF4U\nTwo U/M level sciences (Biology ...,IB Diploma with a minimum of 26 points\nMinimu...,English 12\nPre-Calculus 12\nTwo other approve...,AP English Language and Composition (AP12ELC):...,IELTS: 6.0\nNo band lower than 6.0,TOEFL: 79,Duolingo: 110
1,Algoma University,Environmental Science,ENG4U\nMHF4U\nTwo U/M level sciences (Biology ...,IB Diploma with a minimum of 26 points\nMinimu...,English Studies 12 or English First Peoples 12...,AP English Language and Composition (AP12ELC):...,IELTS: 6.0\nNo band lower than 6.0,TOEFL: 79,Duolingo: 110
2,Algoma University,Computer Science,ENG4U\nMHF4U\nOne additional math course\nMini...,IB Diploma with a minimum of 26 points\nMinimu...,English Studies 12 or English First Peoples 12...,AP English Language and Composition (AP12ELC):...,IELTS: 6.0\nNo band lower than 6.0,TOEFL: 79,Duolingo: 110
3,Algoma University,English,ENG4U\nMinimum average: 70%,IB Diploma with a minimum of 26 points,English Studies 12 or English First Peoples 12...,AP English Language and Composition (AP12ELC):...,IELTS: 6.0\nNo band lower than 6.0,TOEFL: 79\n,Duolingo: 110
4,Algoma University,History,ENG4U\nOne additional U/M course in Canadian a...,IB Diploma with a minimum of 26 points\nMinimu...,English Studies 12 or English First Peoples 12...,AP English Language and Composition (AP12ELC):...,IELTS: 6.0\nNo band lower than 6.0,TOEFL: 79,Duolingo: 110


In [20]:
program = df.iloc[19].to_json()

In [29]:
from pprint import pp

In [30]:
pp(program)

('{"University":"University of Toronto\\nSt. George Campus (Main '
 'Campus)","Program":"Applied Mathematics","ON Admission '
 'Requirement":"ENG4U\\nMHF4U\\nMCV4U\\nTwo of: SBI4U, SCH4U, SPH4U, '
 'ICS4U\\nMinimum average: mid-80s","IB Admission Requirement":"An overall '
 'score of 30\\nIB Mathematics (HL or SL) is required","BC Admission '
 'Requirement":"English (ENG4U)\\nCalculus and Vectors (MCV4U)\\nAdvanced '
 'Functions (MHF4U)\\nTwo of: Biology (SBI4U), Chemistry (SCH4U), Physics '
 '(SPH4U)\\nMinimum average: mid-80s","AP Admission Requirement":"AP English '
 'Language and Composition (AP12ELC): 4+\\nAP Calculus AB (AP12CA): 4+\\nAP '
 'Calculus BC (AP12CB): 4+\\nAP Statistics (AP12SLC): 4+\\nAP Biology '
 '(AP12BI): 4+\\nAP Chemistry (AP12CH): 4+\\nAP Physics 1 (AP12P1): 4+\\nAP '
 'Physics 2 (AP12P2): 4+\\nAP Physics C: Electricity and Magnetism (AP12PEM): '
 '4+\\nAP Physics C: Mechanics (AP12PM): 4+","IELTS":"IELTS: 6.5\\nNo band '
 'below 6.0","TOEFL":"TOEFL: 100\\nMini

In [22]:
print(type(profile1["student_profile_language_tests"]))

<class 'list'>


In [35]:
# profile1["student_profile_language_tests"]
print(
    suggest_on_course(
        {"language": profile1["student_profile_language_tests"], "courses": profile1["student_profile_courses"]},
        program,
    )
)

5
G12 English, ON12EN
G12 Data Management, ON12DM
G12 Biology, ON12BI
G12 Chemistry, ON12CH
G12 Physics, ON12PH
